# GitHub Pull Requests 匯出成 JSONL

這個 Notebook 會抓取指定 GitHub 公開 repository 的 Pull Requests，並輸出成 JSONL。

每筆資料至少包含以下欄位：
- `url`
- `title`
- `pr_description`
- `is_closed`
- `is_approved`
- `comments`（陣列，含發文者 `poster`）
- `last_commit_gitdiff`

## 使用方式
1. 在下一個 cell 設定 `PULLS_URL` 或 `REPO`。
2. 若有 GitHub token，建議設定環境變數 `GITHUB_TOKEN` 以避免 rate limit。
3. 執行全部 cells。
4. 輸出檔案會在 `data/exports/*.jsonl`。

In [ ]:
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from urllib.parse import urlparse

import requests

# ===== 使用者可調整參數 =====
PULLS_URL = "https://github.com/spring-projects/spring-ai/pulls"
REPO: Optional[str] = None  # 例如: "spring-projects/spring-ai"。若有值會覆蓋 PULLS_URL
STATE = "all"              # open | closed | all
MAX_PRS = 30                # None 代表抓全部；建議先用小數量測試
OUTPUT_DIR = Path("data/exports")
SLEEP_SECONDS = 0.1         # 避免太密集請求

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


def parse_repo(repo: Optional[str], pulls_url: str) -> str:
    """把 owner/repo 或 pulls URL 轉成 owner/repo。"""
    if repo:
        if re.fullmatch(r"[^/]+/[^/]+", repo.strip()):
            return repo.strip()
        raise ValueError("REPO 格式需為 'owner/repo'。")

    parsed = urlparse(pulls_url)
    if parsed.netloc not in {"github.com", "www.github.com"}:
        raise ValueError("PULLS_URL 必須是 github.com 網址。")

    parts = [p for p in parsed.path.split("/") if p]
    if len(parts) < 2:
        raise ValueError("PULLS_URL 格式錯誤，應類似 https://github.com/owner/repo/pulls")

    return f"{parts[0]}/{parts[1]}"


OWNER_REPO = parse_repo(REPO, PULLS_URL)
OWNER, REPO_NAME = OWNER_REPO.split("/", 1)
BASE_API = f"https://api.github.com/repos/{OWNER}/{REPO_NAME}"

headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

session = requests.Session()
session.headers.update(headers)

print(f"Target repo: {OWNER_REPO}")
print(f"Token set: {'yes' if GITHUB_TOKEN else 'no'}")

: 

In [ ]:
def gh_get(url: str, params: Optional[Dict] = None, accept: Optional[str] = None, raw_text: bool = False):
    """單次 GitHub API 請求。"""
    local_headers = {}
    if accept:
        local_headers["Accept"] = accept

    resp = session.get(url, params=params, headers=local_headers or None, timeout=30)

    # 基本 rate limit 訊息
    if resp.status_code == 403 and "rate limit" in resp.text.lower():
        reset_at = resp.headers.get("X-RateLimit-Reset")
        raise RuntimeError(f"Rate limit exceeded. X-RateLimit-Reset={reset_at}")

    resp.raise_for_status()
    if raw_text:
        return resp.text
    return resp.json()


def fetch_paginated(url: str, params: Optional[Dict] = None, max_items: Optional[int] = None) -> List[Dict]:
    """抓取 GitHub 分頁資料。"""
    results: List[Dict] = []
    page = 1
    per_page = 100

    while True:
        q = dict(params or {})
        q.update({"page": page, "per_page": per_page})
        batch = gh_get(url, params=q)

        if not isinstance(batch, list) or len(batch) == 0:
            break

        for item in batch:
            results.append(item)
            if max_items is not None and len(results) >= max_items:
                return results

        if len(batch) < per_page:
            break

        page += 1
        if SLEEP_SECONDS > 0:
            time.sleep(SLEEP_SECONDS)

    return results


def compute_is_approved(reviews: List[Dict]) -> bool:
    """以每位 reviewer 最後狀態判斷是否仍存在有效 APPROVED。"""
    latest_state_by_user: Dict[str, str] = {}

    ordered_reviews = sorted(reviews, key=lambda r: r.get("submitted_at") or "")
    for rv in ordered_reviews:
        user = (rv.get("user") or {}).get("login")
        state = (rv.get("state") or "").upper()
        if user:
            latest_state_by_user[user] = state

    return any(state == "APPROVED" for state in latest_state_by_user.values())


def collect_comments(issue_comments: List[Dict], review_comments: List[Dict], reviews: List[Dict]) -> List[Dict]:
    """統一整理訊息格式，並標記 poster。"""
    comments: List[Dict] = []

    for c in issue_comments:
        comments.append(
            {
                "type": "issue_comment",
                "poster": (c.get("user") or {}).get("login"),
                "created_at": c.get("created_at"),
                "url": c.get("html_url"),
                "body": c.get("body") or "",
            }
        )

    for c in review_comments:
        comments.append(
            {
                "type": "review_comment",
                "poster": (c.get("user") or {}).get("login"),
                "created_at": c.get("created_at"),
                "url": c.get("html_url"),
                "body": c.get("body") or "",
            }
        )

    for rv in reviews:
        comments.append(
            {
                "type": "review_event",
                "poster": (rv.get("user") or {}).get("login"),
                "created_at": rv.get("submitted_at"),
                "url": rv.get("html_url"),
                "state": rv.get("state"),
                "body": rv.get("body") or "",
            }
        )

    comments.sort(key=lambda m: m.get("created_at") or "")
    return comments


def fetch_last_commit_diff(pr_number: int) -> str:
    commits_url = f"{BASE_API}/pulls/{pr_number}/commits"
    commits = fetch_paginated(commits_url)
    if not commits:
        return ""

    last_sha = commits[-1].get("sha")
    if not last_sha:
        return ""

    commit_api = f"{BASE_API}/commits/{last_sha}"
    # 使用 diff 媒體類型直接拿 unified diff
    return gh_get(commit_api, accept="application/vnd.github.v3.diff", raw_text=True)


def build_pr_record(pr: Dict) -> Dict:
    pr_number = pr["number"]

    issue_comments = fetch_paginated(f"{BASE_API}/issues/{pr_number}/comments")
    review_comments = fetch_paginated(f"{BASE_API}/pulls/{pr_number}/comments")
    reviews = fetch_paginated(f"{BASE_API}/pulls/{pr_number}/reviews")
    last_commit_gitdiff = fetch_last_commit_diff(pr_number)

    return {
        "url": pr.get("html_url"),
        "title": pr.get("title") or "",
        "pr_description": pr.get("body") or "",
        "is_closed": (pr.get("state") or "").lower() == "closed",
        "is_approved": compute_is_approved(reviews),
        "comments": collect_comments(issue_comments, review_comments, reviews),
        "last_commit_gitdiff": last_commit_gitdiff,
    }


def fetch_pull_requests(state: str = "all", max_prs: Optional[int] = None) -> List[Dict]:
    pulls = fetch_paginated(f"{BASE_API}/pulls", params={"state": state}, max_items=max_prs)
    records: List[Dict] = []

    for idx, pr in enumerate(pulls, start=1):
        print(f"[{idx}/{len(pulls)}] Processing PR #{pr['number']} - {pr.get('title', '')}")
        record = build_pr_record(pr)
        records.append(record)
        if SLEEP_SECONDS > 0:
            time.sleep(SLEEP_SECONDS)

    return records

In [ ]:
records = fetch_pull_requests(state=STATE, max_prs=MAX_PRS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_name = f"{OWNER}_{REPO_NAME}_prs_{STATE}.jsonl"
out_path = OUTPUT_DIR / out_name

with out_path.open("w", encoding="utf-8") as f:
    for item in records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Exported {len(records)} pull requests")
print(f"JSONL path: {out_path}")

# 預覽前 1 筆
if records:
    sample = dict(records[0])
    sample["last_commit_gitdiff"] = (sample.get("last_commit_gitdiff") or "")[:600] + "..."
    print(json.dumps(sample, ensure_ascii=False, indent=2)[:2000])